In [1]:
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import LabelEncoder, StandardScaler
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
# import lightgbm as lgb
# from sklearn.cluster import KMeans
# from sklearn.metrics import silhouette_score
# from scipy.stats import linregress
# import warnings
# warnings.filterwarnings("ignore")

# # ==============================
# # GLOBAL TIME WINDOWS
# # ==============================
# OBSERVATION_START = pd.Timestamp("2015-07-01")
# OBSERVATION_END   = pd.Timestamp("2017-06-30")
# GAP_START         = pd.Timestamp("2017-07-01")
# PREDICTION_START  = pd.Timestamp("2017-08-01")
# PREDICTION_END    = pd.Timestamp("2018-01-31")

# # ==============================
# # 1. DATA LOADING & CLEANING
# # ==============================
# def load_data(history_path="Airline customer churn project/Customer Loyalty History.csv",
#               activity_path="Airline customer churn project/Customer Flight Activity.csv"):
#     hist = pd.read_csv(history_path)
#     flight = pd.read_csv(activity_path)
#     return hist, flight

# def clean_data(hist, flight):
#     hist["Enrollment Date"] = pd.to_datetime(
#         hist["Enrollment Year"].astype(str) + "-" +
#         hist["Enrollment Month"].astype(str).str.zfill(2) + "-01",
#         errors="coerce"
#     )
#     mask_cancel = hist["Cancellation Year"].notna() & hist["Cancellation Month"].notna()
#     hist.loc[mask_cancel, "Cancellation Date"] = pd.to_datetime(
#         hist.loc[mask_cancel, "Cancellation Year"].astype(int).astype(str) + "-" +
#         hist.loc[mask_cancel, "Cancellation Month"].astype(int).astype(str).str.zfill(2) + "-01",
#         errors="coerce"
#     )
#     hist["Cancellation Date"] = pd.to_datetime(hist["Cancellation Date"], errors="coerce")

#     flight["Activity Date"] = pd.to_datetime(
#         flight["Year"].astype(str) + "-" +
#         flight["Month"].astype(str).str.zfill(2) + "-01",
#         errors="coerce"
#     )

#     # Salary imputation
#     hist["Salary"] = hist.groupby(["Education", "Loyalty Card"])["Salary"].transform(
#         lambda x: x.fillna(x.median())
#     )
#     hist["Salary"] = hist["Salary"].fillna(hist["Salary"].median())

#     # Anomalies
#     for col in ["Points Accumulated", "Points Redeemed", "Dollar Cost Points Redeemed"]:
#         flight[col] = flight[col].clip(lower=0)

#     anomaly_mask = (flight["Total Flights"] > 0) & (flight["Distance"] == 0)
#     if anomaly_mask.any():
#         print(f"Warning: {anomaly_mask.sum()} rows with Flights>0 and Distance==0. Imputing Distance.")
#     flight.loc[anomaly_mask, "Distance"] = np.nan
#     flight["Distance"] = flight.groupby("Loyalty Number")["Distance"].transform(
#         lambda x: x.fillna(x.median())
#     )
#     flight["Distance"] = flight["Distance"].fillna(0)

#     return hist, flight

# # ==============================
# # 2. HELPERS
# # ==============================
# def build_monthly_time_series(customer_id, flight_df, start, end):
#     months = pd.date_range(start=start, end=end, freq="MS")
#     df_cust = flight_df[(flight_df["Loyalty Number"] == customer_id) &
#                         (flight_df["Activity Date"] >= start) &
#                         (flight_df["Activity Date"] <= end)].copy()
#     if df_cust.empty:
#         return pd.DataFrame({
#             "Date": months,
#             "Total Flights": 0,
#             "Points Accumulated": 0,
#             "Points Redeemed": 0,
#         })
#     monthly = df_cust.groupby(pd.Grouper(key="Activity Date", freq="MS")).agg({
#         "Total Flights": "sum",
#         "Points Accumulated": "sum",
#         "Points Redeemed": "sum",
#     }).reset_index().rename(columns={"Activity Date": "Date"})
#     full = pd.DataFrame({"Date": months})
#     monthly = full.merge(monthly, on="Date", how="left").fillna(0)
#     return monthly

# def safe_linregress_slope(series):
#     if len(series) < 2 or series.std() == 0:
#         return 0.0
#     slope, _, _, _, _ = linregress(np.arange(len(series)), series.values)
#     return slope

# def compute_gini(array):
#     array = np.array(array, dtype=np.float64)
#     if np.amin(array) < 0:
#         array -= np.amin(array)
#     array = np.sort(array)
#     n = array.shape[0]
#     index = np.arange(1, n + 1)
#     return (2 * np.sum(index * array) - (n + 1) * np.sum(array)) / (n * np.sum(array) + 1e-10)

# # ==============================
# # 3. FEATURE ENGINEERING (OBSERVATION WINDOW)
# # ==============================
# def engineer_features(customers, flight_df):
#     features = []
#     card_order = {"Star": 0, "Nova": 1, "Aurora": 2}
#     customers["Card Tier"] = customers["Loyalty Card"].map(card_order)
#     salary_quartiles = pd.qcut(customers["Salary"], q=4, labels=False, duplicates="drop")

#     for _, row in customers.iterrows():
#         lnum = row["Loyalty Number"]
#         ts = build_monthly_time_series(lnum, flight_df, OBSERVATION_START, OBSERVATION_END)
#         flights = ts["Total Flights"].values
#         points_acc = ts["Points Accumulated"].values
#         points_red = ts["Points Redeemed"].values

#         # Flight Frequency
#         avg_full = flights.mean() if len(flights) > 0 else 0
#         avg_last3 = flights[-3:].mean() if len(flights) >= 3 else avg_full
#         flight_velocity_ratio = avg_last3 / avg_full if avg_full > 0 else 0.0

#         flight_trend_slope = safe_linregress_slope(pd.Series(flights[-12:])) if len(flights) >= 12 else 0.0

#         active_months = np.where(flights[-12:] > 0)[0]
#         gap_acceleration = safe_linregress_slope(pd.Series(np.diff(active_months))) if len(active_months) >= 2 else 0.0

#         # Points Economy
#         history = flight_df[(flight_df["Loyalty Number"] == lnum) &
#                             (flight_df["Activity Date"] <= OBSERVATION_END)]
#         cum_acc = history["Points Accumulated"].sum()
#         cum_red = history["Points Redeemed"].sum()
#         redemption_ratio = cum_red / cum_acc if cum_acc > 0 else 0.0

#         earn_burn_divergence = (safe_linregress_slope(pd.Series(points_acc[-6:])) -
#                                 safe_linregress_slope(pd.Series(points_red[-6:]))) if len(flights) >= 6 else 0.0

#         redemptions = points_red > 0
#         redemption_recency = (len(flights) - 1 - np.where(redemptions)[0][-1]) if redemptions.any() else 999

#         # Seasonal
#         ts["Quarter"] = ts["Date"].dt.quarter
#         quarter_flights = ts.groupby("Quarter")["Total Flights"].sum().reindex([1,2,3,4], fill_value=0)
#         seasonal_gini = compute_gini(quarter_flights.values)

#         ts["Year"] = ts["Date"].dt.year
#         year_q = ts.groupby(["Year", "Quarter"])["Total Flights"].sum().reset_index()
#         year_q["year_total"] = year_q.groupby("Year")["Total Flights"].transform("sum")
#         year_q["dominance"] = year_q["Total Flights"] / year_q["year_total"].replace(0, 1e-10)
#         peak_quarter_dominance = year_q["dominance"].max() if not year_q.empty else 0.0

#         # Demographics interactions
#         sal_quartile = salary_quartiles.loc[row.name]
#         card_tier = row["Card Tier"]
#         mismatch = 1 if ((sal_quartile == 3 and card_tier == 0) or (sal_quartile == 0 and card_tier == 2)) else 0

#         edu_order = {"High School": 0, "Bachelor": 1, "Master": 2, "Doctor": 3}
#         edu_encoded = edu_order.get(row["Education"], 0)
#         education_x_velocity = edu_encoded * flight_velocity_ratio

#         feat = {
#             "Loyalty Number": lnum,
#             "flight_velocity_ratio": flight_velocity_ratio,
#             "flight_trend_slope": flight_trend_slope,
#             "inter_flight_gap_acceleration": gap_acceleration,
#             "redemption_ratio": redemption_ratio,
#             "earn_burn_velocity_divergence": earn_burn_divergence,
#             "redemption_recency": redemption_recency,
#             "seasonal_gini": seasonal_gini,
#             "peak_quarter_dominance": peak_quarter_dominance,
#             "salary_tier_card_mismatch": mismatch,
#             "education_x_velocity": education_x_velocity,
#             "CLV": row["CLV"],
#             "Salary": row["Salary"],
#             "Card Tier": card_tier,
#             "Enrollment Date": row["Enrollment Date"],
#             "Cancellation Date": row["Cancellation Date"] if pd.notna(row["Cancellation Date"]) else pd.NaT,
#         }
#         features.append(feat)

#     feature_df = pd.DataFrame(features).set_index("Loyalty Number")

#     # Tenure
#     months_diff = ((OBSERVATION_END.year - feature_df["Enrollment Date"].dt.year) * 12 +
#                    (OBSERVATION_END.month - feature_df["Enrollment Date"].dt.month))
#     feature_df["tenure_months"] = months_diff.clip(lower=0)

#     # Points balance
#     feature_df["points_balance"] = feature_df.apply(
#         lambda r: flight_df[(flight_df["Loyalty Number"] == r.name) &
#                             (flight_df["Activity Date"] <= OBSERVATION_END)]
#                   .pipe(lambda g: g["Points Accumulated"].sum() - g["Points Redeemed"].sum()),
#         axis=1
#     )
#     return feature_df

# # ==============================
# # 4. CHURN LABELING (TIGHTENED)
# # ==============================
# def label_churn(customers, flight_df):
#     STAGNATION_START = PREDICTION_END - pd.DateOffset(months=17)
#     STAGNATION_END   = PREDICTION_END
#     results = []
#     for lnum in customers["Loyalty Number"]:
#         # Condition 1: Ghosting
#         pred_mask = (flight_df["Loyalty Number"] == lnum) & \
#                     (flight_df["Activity Date"] >= PREDICTION_START) & \
#                     (flight_df["Activity Date"] <= PREDICTION_END)
#         pred_df = flight_df[pred_mask]
#         ghost = (pred_df["Total Flights"].sum() == 0 and
#                  pred_df["Points Accumulated"].sum() == 0 and
#                  pred_df["Points Redeemed"].sum() == 0)

#         # Condition 2: Drop-off
#         ts_pred = build_monthly_time_series(lnum, flight_df, PREDICTION_START, PREDICTION_END)
#         flights_pred = ts_pred["Total Flights"].values
#         ts_obs = build_monthly_time_series(lnum, flight_df, OBSERVATION_START, OBSERVATION_END)
#         baseline = ts_obs["Total Flights"].mean()
#         dropoff = False
#         if len(flights_pred) >= 3:
#             rolled = pd.Series(flights_pred).rolling(window=3, min_periods=1).mean().values
#             below = rolled < (0.3 * baseline)
#             for i in range(len(below) - 1):
#                 if below[i] and below[i + 1]:
#                     dropoff = True
#                     break

#         # Condition 3: Credit-Card-Only Stagnation (tightened)
#         ts_stag = build_monthly_time_series(lnum, flight_df, STAGNATION_START, STAGNATION_END)
#         stag_zero = (ts_stag["Total Flights"].sum() == 0 and ts_stag["Points Redeemed"].sum() == 0)

#         before_stag = flight_df[(flight_df["Loyalty Number"] == lnum) &
#                                 (flight_df["Activity Date"] < STAGNATION_START)]
#         lifetime_flights_before = before_stag["Total Flights"].sum()

#         full_hist = flight_df[(flight_df["Loyalty Number"] == lnum) &
#                               (flight_df["Activity Date"] <= PREDICTION_END)]
#         balance = full_hist["Points Accumulated"].sum() - full_hist["Points Redeemed"].sum()

#         stagnation = stag_zero and (lifetime_flights_before > 0) and (balance > 15000)

#         churn = ghost or dropoff or stagnation
#         results.append({
#             "Loyalty Number": lnum,
#             "ghost": ghost,
#             "dropoff": dropoff,
#             "stagnation": stagnation,
#             "is_churned": churn
#         })

#     return pd.DataFrame(results).set_index("Loyalty Number")

# # ==============================
# # 5. MODELING & VALIDATION
# # ==============================
# def train_evaluate(features, target_series):
#     drop_cols = ["Enrollment Date", "Cancellation Date"]
#     X = features.drop(columns=drop_cols, errors="ignore")
#     y = target_series.loc[X.index]

#     for col in X.select_dtypes(include="object"):
#         le = LabelEncoder()
#         X[col] = le.fit_transform(X[col].astype(str))

#     scaler = StandardScaler()
#     X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

#     pos = y.sum()
#     neg = len(y) - pos
#     scale_pos_weight = neg / pos if pos > 0 else 1.0

#     lgb_params = {
#         "objective": "binary",
#         "metric": "auc",
#         "boosting_type": "gbdt",
#         "scale_pos_weight": scale_pos_weight,
#         "num_leaves": 31,
#         "learning_rate": 0.05,
#         "feature_fraction": 0.8,
#         "verbose": -1,
#         "random_state": 42,
#     }

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     oof_preds = np.zeros(len(y))
#     models = []
#     feature_importance = []

#     print("Starting 5-fold cross-validation...")
#     for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         model = lgb.LGBMClassifier(**lgb_params, n_estimators=200)
#         model.fit(
#             X_train, y_train,
#             eval_set=[(X_val, y_val)],
#             callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(0)]
#         )
#         oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
#         models.append(model)

#         fold_imp = pd.DataFrame({
#             "feature": X.columns,
#             "importance": model.feature_importances_
#         }).sort_values("importance", ascending=False)
#         feature_importance.append(fold_imp)

#     avg_imp = pd.concat(feature_importance).groupby("feature").mean().sort_values("importance", ascending=False)
#     roc_auc = roc_auc_score(y, oof_preds)
#     pr_auc = average_precision_score(y, oof_preds)
#     pred_class = (oof_preds >= 0.5).astype(int)
#     cm = confusion_matrix(y, pred_class)

#     # Final model on all data
#     final_model = lgb.LGBMClassifier(**lgb_params, n_estimators=200)
#     final_model.fit(X, y)

#     return final_model, scaler, X.columns.tolist(), X_scaled, avg_imp, roc_auc, pr_auc, cm, classification_report(y, pred_class)

# # ==============================
# # 6. CLUSTERING & PERSONAS
# # ==============================
# def perform_clustering(X_scaled, max_k=10):
#     scores = []
#     K_range = range(2, min(max_k + 1, len(X_scaled)))
#     for k in K_range:
#         km = KMeans(n_clusters=k, random_state=42, n_init=10)
#         labels = km.fit_predict(X_scaled)
#         scores.append(silhouette_score(X_scaled, labels))
#     best_k = K_range[np.argmax(scores)]
#     km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
#     clusters = km.fit_predict(X_scaled)
#     return clusters, best_k, km

# def persona_profiling(features, clusters):
#     df = features.copy()
#     df["cluster"] = clusters
#     cluster_stats = df.groupby("cluster").agg({
#         "CLV": "mean",
#         "Salary": "mean",
#         "Card Tier": "mean",
#         "flight_velocity_ratio": "mean",
#         "seasonal_gini": "mean",
#         "redemption_ratio": "mean",
#         "flight_trend_slope": "mean",
#         "tenure_months": "mean",
#         "points_balance": "mean",
#     }).round(2)

#     def classify(row):
#         clv = row["CLV"]
#         salary = row["Salary"]
#         card = row["Card Tier"]
#         freq_ratio = row["flight_velocity_ratio"]
#         gini = row["seasonal_gini"]
#         red_ratio = row["redemption_ratio"]
#         slope = row["flight_trend_slope"]
#         tenure = row["tenure_months"]
#         bal = row["points_balance"]

#         if clv > 7000 and salary > 80000 and card >= 1 and freq_ratio > 0.8 and gini < 0.3 and 0.25 < red_ratio < 0.55:
#             return "Crown Jewel Business Flyer"
#         if bal > 50000 and red_ratio < 0.15:
#             return "Points Hoarder"
#         if clv < 3000 and card < 1 and gini > 0.7:
#             return "Seasonal Vacationer"
#         if clv > 5000 and card >= 1 and slope < 0 and freq_ratio < 0.35:
#             return "Fading Gold"
#         if clv < 4000 and tenure < 36 and freq_ratio > 1.2:
#             return "Rising Star"
#         return "Unassigned"

#     persona_map = cluster_stats.apply(classify, axis=1).to_dict()
#     # Assign each customer a persona
#     df["persona"] = df["cluster"].map(persona_map)
#     persona_counts = df["persona"].value_counts()
#     # Ensure all 5 personas appear
#     for p in ["Crown Jewel Business Flyer", "Points Hoarder", "Seasonal Vacationer",
#               "Fading Gold", "Rising Star"]:
#         if p not in persona_counts:
#             persona_counts[p] = 0
#     persona_counts = persona_counts[["Crown Jewel Business Flyer", "Points Hoarder",
#                                      "Seasonal Vacationer", "Fading Gold", "Rising Star"]]
#     return persona_map, cluster_stats, persona_counts

# # ==============================
# # 7. MAIN PIPELINE
# # ==============================
# if __name__ == "__main__":
#     print("Loading data...")
#     hist, flight = load_data(
#         history_path="Airline customer churn project/Customer Loyalty History.csv",
#         activity_path="Airline customer churn project/Customer Flight Activity.csv"
#     )
#     hist, flight = clean_data(hist, flight)

#     print("\nEngineering features (this may take several minutes)...")
#     features = engineer_features(hist, flight)

#     print("Labeling churn...")
#     target_df = label_churn(hist, flight)

#     # Churn rate
#     churn_rate = target_df["is_churned"].mean()
#     print(f"\nChurn Rate in data: {churn_rate:.2%}")

#     target = target_df["is_churned"]
#     data = features.join(target, how="inner")

#     print("\nTraining model...")
#     final_model, scaler, feature_cols, X_scaled, avg_imp, roc_auc, pr_auc, cm, report = train_evaluate(
#         data.drop(columns=["is_churned"]), data["is_churned"]
#     )

#     print(f"\nModel ROC-AUC: {roc_auc:.4f}")
#     print(f"Model PR-AUC: {pr_auc:.4f}")

#     # Top 3 features
#     top3 = avg_imp.head(3).index.tolist()
#     print(f"Top 3 Churn Features: {top3}")

#     # Clustering
#     print("\nPerforming customer segmentation...")
#     clusters, best_k, km = perform_clustering(X_scaled, max_k=10)
#     persona_map, cluster_stats, persona_counts = persona_profiling(data, clusters)

#     print(f"\nOptimal number of clusters: {best_k}")
#     print("Cluster Centroids:\n", cluster_stats)
#     print("\nCluster -> Persona Mapping:")
#     for k, v in persona_map.items():
#         print(f"  Cluster {k}: {v}")

#     print("\nCluster sizes (members per persona):")
#     for persona, count in persona_counts.items():
#         print(f"  {persona}: {count}")

Loading data...

Engineering features (this may take several minutes)...
Labeling churn...

Churn Rate in data: 39.29%

Training model...
Starting 5-fold cross-validation...
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[4]	valid_0's auc: 0.754321
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[27]	valid_0's auc: 0.779761
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.775429
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.778542
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[16]	valid_0's auc: 0.754837

Model ROC-AUC: 0.7546
Model PR-AUC: 0.6841
Top 3 Churn Features: ['tenure_months', 'earn_burn_velocity_divergence', 'Salary']

Performing customer segmentation...

Optimal number of clusters: 2
Cluster C

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.stats import linregress
import warnings
warnings.filterwarnings("ignore")

# ==============================
# GLOBAL TIME WINDOWS
# ==============================
OBSERVATION_START = pd.Timestamp("2015-07-01")
OBSERVATION_END   = pd.Timestamp("2017-06-30")
GAP_START         = pd.Timestamp("2017-07-01")
PREDICTION_START  = pd.Timestamp("2017-08-01")
PREDICTION_END    = pd.Timestamp("2018-01-31")

# ==============================
# 1. DATA LOADING & CLEANING
# ==============================
def load_data(history_path="customer_loyalty_history.csv",
              activity_path="customer_flight_activity.csv"):
    hist = pd.read_csv(history_path)
    flight = pd.read_csv(activity_path)
    return hist, flight

def clean_data(hist, flight):
    hist["Enrollment Date"] = pd.to_datetime(
        hist["Enrollment Year"].astype(str) + "-" +
        hist["Enrollment Month"].astype(str).str.zfill(2) + "-01",
        errors="coerce"
    )
    mask_cancel = hist["Cancellation Year"].notna() & hist["Cancellation Month"].notna()
    hist.loc[mask_cancel, "Cancellation Date"] = pd.to_datetime(
        hist.loc[mask_cancel, "Cancellation Year"].astype(int).astype(str) + "-" +
        hist.loc[mask_cancel, "Cancellation Month"].astype(int).astype(str).str.zfill(2) + "-01",
        errors="coerce"
    )
    hist["Cancellation Date"] = pd.to_datetime(hist["Cancellation Date"], errors="coerce")

    flight["Activity Date"] = pd.to_datetime(
        flight["Year"].astype(str) + "-" +
        flight["Month"].astype(str).str.zfill(2) + "-01",
        errors="coerce"
    )

    # Salary imputation
    hist["Salary"] = hist.groupby(["Education", "Loyalty Card"])["Salary"].transform(
        lambda x: x.fillna(x.median())
    )
    hist["Salary"] = hist["Salary"].fillna(hist["Salary"].median())

    # Anomalies
    for col in ["Points Accumulated", "Points Redeemed", "Dollar Cost Points Redeemed"]:
        flight[col] = flight[col].clip(lower=0)

    anomaly_mask = (flight["Total Flights"] > 0) & (flight["Distance"] == 0)
    if anomaly_mask.any():
        print(f"Warning: {anomaly_mask.sum()} rows with Flights>0 and Distance==0. Imputing Distance.")
    flight.loc[anomaly_mask, "Distance"] = np.nan
    flight["Distance"] = flight.groupby("Loyalty Number")["Distance"].transform(
        lambda x: x.fillna(x.median())
    )
    flight["Distance"] = flight["Distance"].fillna(0)

    return hist, flight

# ==============================
# 2. HELPERS
# ==============================
def build_monthly_time_series(customer_id, flight_df, start, end):
    months = pd.date_range(start=start, end=end, freq="MS")
    df_cust = flight_df[(flight_df["Loyalty Number"] == customer_id) &
                        (flight_df["Activity Date"] >= start) &
                        (flight_df["Activity Date"] <= end)].copy()
    if df_cust.empty:
        return pd.DataFrame({
            "Date": months,
            "Total Flights": 0,
            "Points Accumulated": 0,
            "Points Redeemed": 0,
        })
    monthly = df_cust.groupby(pd.Grouper(key="Activity Date", freq="MS")).agg({
        "Total Flights": "sum",
        "Points Accumulated": "sum",
        "Points Redeemed": "sum",
    }).reset_index().rename(columns={"Activity Date": "Date"})
    full = pd.DataFrame({"Date": months})
    monthly = full.merge(monthly, on="Date", how="left").fillna(0)
    return monthly

def safe_linregress_slope(series):
    if len(series) < 2 or series.std() == 0:
        return 0.0
    slope, _, _, _, _ = linregress(np.arange(len(series)), series.values)
    return slope

def compute_gini(array):
    array = np.array(array, dtype=np.float64)
    if np.amin(array) < 0:
        array -= np.amin(array)
    array = np.sort(array)
    n = array.shape[0]
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * array) - (n + 1) * np.sum(array)) / (n * np.sum(array) + 1e-10)

# ==============================
# 3. FEATURE ENGINEERING (OBSERVATION WINDOW)
# ==============================
def engineer_features(customers, flight_df):
    features = []
    card_order = {"Star": 0, "Nova": 1, "Aurora": 2}
    customers["Card Tier"] = customers["Loyalty Card"].map(card_order)
    salary_quartiles = pd.qcut(customers["Salary"], q=4, labels=False, duplicates="drop")

    for _, row in customers.iterrows():
        lnum = row["Loyalty Number"]
        ts = build_monthly_time_series(lnum, flight_df, OBSERVATION_START, OBSERVATION_END)
        flights = ts["Total Flights"].values
        points_acc = ts["Points Accumulated"].values
        points_red = ts["Points Redeemed"].values

        # Flight Frequency
        avg_full = flights.mean() if len(flights) > 0 else 0
        avg_last3 = flights[-3:].mean() if len(flights) >= 3 else avg_full
        flight_velocity_ratio = avg_last3 / avg_full if avg_full > 0 else 0.0

        flight_trend_slope = safe_linregress_slope(pd.Series(flights[-12:])) if len(flights) >= 12 else 0.0

        active_months = np.where(flights[-12:] > 0)[0]
        gap_acceleration = safe_linregress_slope(pd.Series(np.diff(active_months))) if len(active_months) >= 2 else 0.0

        # Points Economy
        history = flight_df[(flight_df["Loyalty Number"] == lnum) &
                            (flight_df["Activity Date"] <= OBSERVATION_END)]
        cum_acc = history["Points Accumulated"].sum()
        cum_red = history["Points Redeemed"].sum()
        redemption_ratio = cum_red / cum_acc if cum_acc > 0 else 0.0

        earn_burn_divergence = (safe_linregress_slope(pd.Series(points_acc[-6:])) -
                                safe_linregress_slope(pd.Series(points_red[-6:]))) if len(flights) >= 6 else 0.0

        redemptions = points_red > 0
        redemption_recency = (len(flights) - 1 - np.where(redemptions)[0][-1]) if redemptions.any() else 999

        # Seasonal
        ts["Quarter"] = ts["Date"].dt.quarter
        quarter_flights = ts.groupby("Quarter")["Total Flights"].sum().reindex([1,2,3,4], fill_value=0)
        seasonal_gini = compute_gini(quarter_flights.values)

        ts["Year"] = ts["Date"].dt.year
        year_q = ts.groupby(["Year", "Quarter"])["Total Flights"].sum().reset_index()
        year_q["year_total"] = year_q.groupby("Year")["Total Flights"].transform("sum")
        year_q["dominance"] = year_q["Total Flights"] / year_q["year_total"].replace(0, 1e-10)
        peak_quarter_dominance = year_q["dominance"].max() if not year_q.empty else 0.0

        # Demographics interactions
        sal_quartile = salary_quartiles.loc[row.name]
        card_tier = row["Card Tier"]
        mismatch = 1 if ((sal_quartile == 3 and card_tier == 0) or (sal_quartile == 0 and card_tier == 2)) else 0

        edu_order = {"High School": 0, "Bachelor": 1, "Master": 2, "Doctor": 3}
        edu_encoded = edu_order.get(row["Education"], 0)
        education_x_velocity = edu_encoded * flight_velocity_ratio

        feat = {
            "Loyalty Number": lnum,
            "flight_velocity_ratio": flight_velocity_ratio,
            "flight_trend_slope": flight_trend_slope,
            "inter_flight_gap_acceleration": gap_acceleration,
            "redemption_ratio": redemption_ratio,
            "earn_burn_velocity_divergence": earn_burn_divergence,
            "redemption_recency": redemption_recency,
            "seasonal_gini": seasonal_gini,
            "peak_quarter_dominance": peak_quarter_dominance,
            "salary_tier_card_mismatch": mismatch,
            "education_x_velocity": education_x_velocity,
            "CLV": row["CLV"],
            "Salary": row["Salary"],
            "Card Tier": card_tier,
            "Enrollment Date": row["Enrollment Date"],
            "Cancellation Date": row["Cancellation Date"] if pd.notna(row["Cancellation Date"]) else pd.NaT,
        }
        features.append(feat)

    feature_df = pd.DataFrame(features).set_index("Loyalty Number")

    # Tenure
    months_diff = ((OBSERVATION_END.year - feature_df["Enrollment Date"].dt.year) * 12 +
                   (OBSERVATION_END.month - feature_df["Enrollment Date"].dt.month))
    feature_df["tenure_months"] = months_diff.clip(lower=0)

    # Points balance
    feature_df["points_balance"] = feature_df.apply(
        lambda r: flight_df[(flight_df["Loyalty Number"] == r.name) &
                            (flight_df["Activity Date"] <= OBSERVATION_END)]
                  .pipe(lambda g: g["Points Accumulated"].sum() - g["Points Redeemed"].sum()),
        axis=1
    )
    return feature_df

# ==============================
# 4. CHURN LABELING (TIGHTENED)
# ==============================
def label_churn(customers, flight_df):
    STAGNATION_START = PREDICTION_END - pd.DateOffset(months=17)
    STAGNATION_END   = PREDICTION_END
    results = []
    for lnum in customers["Loyalty Number"]:
        # Condition 1: Ghosting
        pred_mask = (flight_df["Loyalty Number"] == lnum) & \
                    (flight_df["Activity Date"] >= PREDICTION_START) & \
                    (flight_df["Activity Date"] <= PREDICTION_END)
        pred_df = flight_df[pred_mask]
        ghost = (pred_df["Total Flights"].sum() == 0 and
                 pred_df["Points Accumulated"].sum() == 0 and
                 pred_df["Points Redeemed"].sum() == 0)

        # Condition 2: Drop-off
        ts_pred = build_monthly_time_series(lnum, flight_df, PREDICTION_START, PREDICTION_END)
        flights_pred = ts_pred["Total Flights"].values
        ts_obs = build_monthly_time_series(lnum, flight_df, OBSERVATION_START, OBSERVATION_END)
        baseline = ts_obs["Total Flights"].mean()
        dropoff = False
        if len(flights_pred) >= 3:
            rolled = pd.Series(flights_pred).rolling(window=3, min_periods=1).mean().values
            below = rolled < (0.3 * baseline)
            for i in range(len(below) - 1):
                if below[i] and below[i + 1]:
                    dropoff = True
                    break

        # Condition 3: Credit-Card-Only Stagnation (tightened)
        ts_stag = build_monthly_time_series(lnum, flight_df, STAGNATION_START, STAGNATION_END)
        stag_zero = (ts_stag["Total Flights"].sum() == 0 and ts_stag["Points Redeemed"].sum() == 0)

        before_stag = flight_df[(flight_df["Loyalty Number"] == lnum) &
                                (flight_df["Activity Date"] < STAGNATION_START)]
        lifetime_flights_before = before_stag["Total Flights"].sum()

        full_hist = flight_df[(flight_df["Loyalty Number"] == lnum) &
                              (flight_df["Activity Date"] <= PREDICTION_END)]
        balance = full_hist["Points Accumulated"].sum() - full_hist["Points Redeemed"].sum()

        stagnation = stag_zero and (lifetime_flights_before > 0) and (balance > 15000)

        churn = ghost or dropoff or stagnation
        results.append({
            "Loyalty Number": lnum,
            "ghost": ghost,
            "dropoff": dropoff,
            "stagnation": stagnation,
            "is_churned": churn
        })

    return pd.DataFrame(results).set_index("Loyalty Number")

# ==============================
# 5. MODELING & VALIDATION
# ==============================
def train_evaluate(features, target_series):
    drop_cols = ["Enrollment Date", "Cancellation Date"]
    X = features.drop(columns=drop_cols, errors="ignore")
    y = target_series.loc[X.index]

    for col in X.select_dtypes(include="object"):
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

    pos = y.sum()
    neg = len(y) - pos
    scale_pos_weight = neg / pos if pos > 0 else 1.0

    lgb_params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "scale_pos_weight": scale_pos_weight,
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.8,
        "verbose": -1,
        "random_state": 42,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(y))
    models = []
    feature_importance = []

    print("Starting 5-fold cross-validation...")
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**lgb_params, n_estimators=200)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(0)]
        )
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        models.append(model)

        fold_imp = pd.DataFrame({
            "feature": X.columns,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
        feature_importance.append(fold_imp)

    avg_imp = pd.concat(feature_importance).groupby("feature").mean().sort_values("importance", ascending=False)
    roc_auc = roc_auc_score(y, oof_preds)
    pr_auc = average_precision_score(y, oof_preds)
    pred_class = (oof_preds >= 0.5).astype(int)
    cm = confusion_matrix(y, pred_class)

    final_model = lgb.LGBMClassifier(**lgb_params, n_estimators=200)
    final_model.fit(X, y)

    return final_model, scaler, X.columns.tolist(), X_scaled, avg_imp, roc_auc, pr_auc, cm, classification_report(y, pred_class)

# ==============================
# 6. CUSTOMER SEGMENTATION (FIXED 5 CLUSTERS, RELATIVE MAPPING)
# ==============================
def perform_clustering(X_scaled):
    """Force K-Means with n_clusters=5."""
    km = KMeans(n_clusters=5, random_state=42, n_init=10)
    clusters = km.fit_predict(X_scaled)
    return clusters, km

def persona_profiling(features, clusters):
    """
    Map 5 clusters to 5 personas using relative centroid characteristics.
    No cluster is left unassigned.
    """
    df = features.copy()
    df["cluster"] = clusters

    # Centroids for key metrics
    centroids = df.groupby("cluster").agg({
        "seasonal_gini": "mean",
        "points_balance": "mean",
        "flight_velocity_ratio": "mean",
        "flight_trend_slope": "mean",
        "CLV": "mean",
        "redemption_ratio": "mean",
        "tenure_months": "mean",
        "Card Tier": "mean",
    })

    persona_map = {}
    remaining = set(centroids.index)

    # 1. Seasonal Vacationer: highest seasonal_gini
    vac_cluster = centroids["seasonal_gini"].idxmax()
    persona_map[vac_cluster] = "Seasonal Vacationer"
    remaining.remove(vac_cluster)

    # 2. Points Hoarder: highest points_balance
    hoard_cluster = centroids.loc[list(remaining), "points_balance"].idxmax()
    persona_map[hoard_cluster] = "Points Hoarder"
    remaining.remove(hoard_cluster)

    # 3. Rising Star: highest flight_velocity_ratio among remaining
    rising_cluster = centroids.loc[list(remaining), "flight_velocity_ratio"].idxmax()
    persona_map[rising_cluster] = "Rising Star"
    remaining.remove(rising_cluster)

    # Now two clusters left: differentiate Fading Gold vs Crown Jewel
    # Fading Gold: most negative flight_trend_slope AND low flight_velocity_ratio
    last_two = list(remaining)
    # Score: slope (lower is more negative) + velocity (lower) → sort by slope ascending + velocity ascending
    # Simple approach: pick the one with the more negative slope as Fading Gold; if tied, lower velocity.
    slopes = centroids.loc[last_two, "flight_trend_slope"]
    if slopes.idxmin() == slopes.idxmax():
        # If equal slopes, use velocity
        velocities = centroids.loc[last_two, "flight_velocity_ratio"]
        fading_cluster = velocities.idxmin()
    else:
        fading_cluster = slopes.idxmin()
    persona_map[fading_cluster] = "Fading Gold"
    remaining.remove(fading_cluster)

    # The last remaining cluster is Crown Jewel Business Flyer
    crown_cluster = remaining.pop()
    persona_map[crown_cluster] = "Crown Jewel Business Flyer"

    # Assign persona to each customer
    df["persona"] = df["cluster"].map(persona_map)
    persona_counts = df["persona"].value_counts()
    # Ensure all 5 appear in output
    for p in ["Crown Jewel Business Flyer", "Points Hoarder", "Seasonal Vacationer",
              "Fading Gold", "Rising Star"]:
        if p not in persona_counts:
            persona_counts[p] = 0
    persona_counts = persona_counts[["Crown Jewel Business Flyer", "Points Hoarder",
                                     "Seasonal Vacationer", "Fading Gold", "Rising Star"]]

    return persona_map, centroids, persona_counts

# ==============================
# 7. MAIN PIPELINE
# ==============================
if __name__ == "__main__":
    print("Loading data...")
    hist, flight = load_data(
        history_path="Airline customer churn project/Customer Loyalty History.csv",
        activity_path="Airline customer churn project/Customer Flight Activity.csv"
    )
    hist, flight = clean_data(hist, flight)

    print("\nEngineering features (this may take several minutes)...")
    features = engineer_features(hist, flight)

    print("Labeling churn...")
    target_df = label_churn(hist, flight)

    churn_rate = target_df["is_churned"].mean()
    print(f"\nChurn Rate in data: {churn_rate:.2%}")

    target = target_df["is_churned"]
    data = features.join(target, how="inner")

    print("\nTraining model...")
    final_model, scaler, feature_cols, X_scaled, avg_imp, roc_auc, pr_auc, cm, report = train_evaluate(
        data.drop(columns=["is_churned"]), data["is_churned"]
    )

    print(f"\nModel ROC-AUC: {roc_auc:.4f}")
    print(f"Model PR-AUC: {pr_auc:.4f}")

    top3 = avg_imp.head(3).index.tolist()
    print(f"Top 3 Churn Features: {top3}")

    print("\nPerforming customer segmentation...")
    clusters, km = perform_clustering(X_scaled)
    persona_map, centroids, persona_counts = persona_profiling(data, clusters)

    print("\nCluster Centroids:\n", centroids.round(2))
    print("\nCluster -> Persona Mapping:")
    for k, v in persona_map.items():
        print(f"  Cluster {k}: {v}")

    print("\nCluster sizes (members per persona):")
    for persona, count in persona_counts.items():
        print(f"  {persona}: {count}")

Loading data...

Engineering features (this may take several minutes)...
Labeling churn...

Churn Rate in data: 39.29%

Training model...
Starting 5-fold cross-validation...
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[4]	valid_0's auc: 0.754321
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[27]	valid_0's auc: 0.779761
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.775429
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.778542
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[16]	valid_0's auc: 0.754837

Model ROC-AUC: 0.7546
Model PR-AUC: 0.6841
Top 3 Churn Features: ['tenure_months', 'earn_burn_velocity_divergence', 'Salary']

Performing customer segmentation...

Cluster Centroids:
          seasonal_g